In [0]:
import json
import requests
from  pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
spark.sql("create catalog if not exists workspace")
spark.sql("create schema if not exists workspace.default")
spark.sql("create volume if not exists workspace.default.cricket_api_project")

In [0]:
base_path="/Volumes/workspace/default/cricket_api_project"

In [0]:
API_KEY='86c9ce88-dbd9-48b7-921a-042ecbfe47e2'
api_url=f"https://api.cricapi.com/v1/countries?apikey={API_KEY}&offset=0"

response=requests.get(api_url)
response.raise_for_status()

api_data=response.json()
print(api_data.keys())

print(json.dumps(api_data,indent=2)[:2000])

In [0]:
raw_file_path=f'{base_path}/current_matches_raw.json'
with open(raw_file_path,'w') as file:
    json.dump(api_data,file)
print("RAW API data is save at the : ", raw_file_path)

In [0]:
bronze_data=[
    {
        "source_api":api_url,
        "raw_data":json.dumps(api_data),
        "ingestion_time":None
    }
]

bronze_schema=StructType([
    StructField("source_api",StringType(),True),
    StructField("raw_data",StringType(),True),
    StructField("ingestion_time", TimestampType(),True)
])

bronze_df=spark.createDataFrame(bronze_data,bronze_schema).withColumn("ingestion_time",current_timestamp())

display(bronze_df)

In [0]:
bronze_df.write\
    .format('delta')\
        .mode('overwrite')\
            .saveAsTable("workspace.default.cricket_api_project")
print("Bronze table created successfully")

In [0]:
%sql
select * from workspace.default.cricket_api_project;